# BENCH-Mobility for Vienna

BENCH is an agent-based model that applies behavioural theory to simulate the adoption of various low carbon actions
This version of BENCH aims to gain a better understanding of active mobility in cities, including needs, opportunities, and pathways.  
- In the first phase of the project, we are conducting this survey to explore individuals mobility preferences. 
- In the second phase, we will use the survey data to develop an empirical agent-based model, designed to provide deeper insights into the 
behavioral and social dynamics, and infrastructural/service provisioning influencing mobility choices. mobility choices.


In [1]:
# Import packages
import os
import sys
from pathlib import Path
import random
import pandas as pd
import numpy as np
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa import DataCollector
from mesa.visualization import SolaraViz, make_plot_component, make_space_component
import configparser
import geopandas as gpd


Could not check jupyter-widgets extensions.
Traceback (most recent call last):
  File "C:\Users\torrenperaire\Documents\Code\BENCH_ActMob\.venv\Lib\site-packages\solara\checks.py", line 201, in check_jupyter
    python_executable = server_python or get_server_python_executable(silent)
                                         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "C:\Users\torrenperaire\Documents\Code\BENCH_ActMob\.venv\Lib\site-packages\solara\checks.py", line 155, in get_server_python_executable
    pythons = [getcmdline(server["pid"]) for server in servers]
               ~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\torrenperaire\Documents\Code\BENCH_ActMob\.venv\Lib\site-packages\solara\checks.py", line 135, in getcmdline
    return subprocess.check_output(["wmic", "process", "get", "commandline", "/format:list"]).split(b"\n")[0].split(b" ")[0].decode("utf-8")
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\torrenper

In [2]:
# Create agent class
class HouseholdAgent(Agent):
    def __init__(self, unique_id, model, **kwargs):
        super().__init__(model)
        
        self.h_id = unique_id
        self.action = 0
        
        for attr,value in kwargs.items():
            try:
                value = int(value)
            except:
                pass
            setattr(self, attr, value)

In [3]:
# Create model class
class CommunityModel(Model):
    def __init__(self, population, vienna_housing_map, coefs, district_infra, start_year = 2018, simulation_start = 2024, num_agents = 100, num_peers = 5, case_study="Vienna", scenario="BA", 
                 data="Empirical-survey", learning="Informative", map_update="No", policy="No", iteration = 0):
        super().__init__()
        self.num_agents = num_agents
        self.num_peers = num_peers
        # self.schedule = RandomActivation(self)
        self.learning = learning
        self.tr_modes = ['car', 'bike', 'walk', 'pt']
        self.dests = ['work', 'services', 'leisure']
        self.grid = MultiGrid(3500, 2500, True)  # Assuming a 3500x2500 grid to represent Vienna
        self.case_study = case_study
        self.scenario = scenario
        self.start_year = start_year
        self.simulation_start = simulation_start
        self.data = data
        self.map_update = map_update
        self.district_infra = district_infra
        self.policy = policy
        self.population = population
        self.vienna_housing_map = vienna_housing_map
        self.year = start_year
        self.iter = iteration
        self.debugfiles = True
        self.awareness_levels = {'Low': [], 'Medium': [], 'High': []}
        self.motivation_levels = {'PN': [], 'SN': []}
        self.behavioral_changes = {'Car': [], 'Public_tr': [], 'Motor': [], 'Bicycle': [], 'Walk': []}
        self.mode_changes = {'I_bicycle': [], 'I_walk': []}
        self.energy_savings = {'Fuel': []}
        self.invs1_com = 0
        self.a1_com = 0
        self.coefs = coefs
        
        # self.I_bicycle_cost = i1_cost
        # self.I_walk_cost = i2_cost
        # self.load_data()
        self.setup_agents()
        self.datacollector = DataCollector(
            model_reporters={"Bicycle": lambda m: sum(agent.mode_trips['bike'] for agent in m.agents) / sum(agent.total_trips for agent in m.agents),
                             "Walk": lambda m: sum(agent.mode_trips['walk'] for agent in m.agents) / sum(agent.total_trips for agent in m.agents),
                             # "Bicycle": lambda m: {d: sum(agent.mode_trips['bike'] for agent in m.agents if agent.district == d) for d in range(1, 24)},
                             # "Walk": lambda m: {d: sum(agent.mode_trips['walk'] for agent in m.agents if agent.district == d) for d in range(1, 24)},
                             # "Utility Bicycle": lambda m: np.mean([agent.U_bike for agent in m.agents]),
                             # "Utility Walk": lambda m: np.mean([agent.U_walk for agent in m.agents]),
                             "Awareness": lambda m: np.mean([agent.kn_aw for agent in m.agents]),
                             "PersonalNorm": lambda m: np.mean([agent.pn for agent in m.agents]),
                             "PerceivedBehaviourControl": lambda m: np.mean([agent.pbc for agent in m.agents])}
        )
        self.running = True
        self.datacollector.collect(self)


    # def load_data(self):
    #     try:
    #         self.survey = pd.read_csv('input//survey_encoded.csv', index_col = 0)
    #         print(f"Data loaded successfully for case study {self.case_study} and scenario {self.scenario}.")
    #     except FileNotFoundError as e:
    #         print(f"File not found: {e.filename}")

    def setup_agents(self):
        if self.case_study == "Vienna" and self.data == "Empirical-survey":
            # Import data
            setup_vars = pd.read_csv('input//agent_setup.csv', index_col = 0)
            setup_vars_dict = setup_vars.to_dict()
            # Housing map
            distances = ['bike_dist', 'green_dist', 'hike_dist', 'resid_dist', 'slow_dist']
            facilities = ['bike_bool', 'green_bool', 'hike_bool', 'resid_bool', 'slow_bool']
            # Distances between houses and facilities
            housing_distances = self.vienna_housing_map[distances]
            housing_facilities = self.vienna_housing_map[facilities]
            # Subdistricts
            housing_subdistrict = self.vienna_housing_map['ZBEZ'].astype(int)
            # House coordinates by district
            coordinates = {}
            idxs = {}
            probs = {}
            distr_houses = {}
            for distr, houses in self.vienna_housing_map.groupby('BEZIRK'):
                distr_houses[distr] = houses
                idxs[distr] = houses.index
                distr_coords = houses.get_coordinates()
                coordinates[distr] = distr_coords[~distr_coords.index.duplicated(keep='first')]
                coordinates[distr].index = houses.index
                probs[distr] = houses.WOHNUNGSAN.fillna(0) / sum(houses.WOHNUNGSAN.fillna(0))
            # Collect the indices of the agents in the different clusters
            cluster_dict  = {cluster: list() for cluster in self.population.cluster.unique()}
            
            # Create agents
            for i in range(self.num_agents):
                agent = HouseholdAgent(i, self, **setup_vars_dict['Value'])
                # Assign attributes based on simulated population
                self.assign_attributes(agent, self.population, setup_vars)
                district = agent.district
                # Randomly allocate agents to a social house
                idx = np.random.choice(idxs[district], 1, p = probs[district])
                coords = list(zip(coordinates[district].x, coordinates[district].y))
                agent.district = int(district)
                agent.subdistrict = housing_subdistrict.loc[idx]
                agent.house_id = idx
                # Adjust coordinates to match the multigrid
                list_idx = idxs[district].get_loc(idx[0])
                x = np.round((coords[list_idx][0] - 16.2) * 10000, 0)
                y = np.round((coords[list_idx][1] - 48.1) * 10000, 0)
                self.grid.place_agent(agent, (int(x), int(y)))
                # Add houseing characteristics to agent (distances from facilities)
                agent.distances = housing_distances.loc[idx].to_dict(orient = 'list')
                agent.facilities = housing_facilities.loc[idx].to_dict(orient = 'list')
                # Add to corresponding cluster
                cluster_dict[agent.cluster] = cluster_dict[agent.cluster] + [i]
                # Randomly select when the agent shifted mode the last time
                agent.shift_year = random.randint(0, 3)
                # Assess trips
                agent.modes = {'work': agent.mode_work, 'services': agent.mode_services, 'leisure': agent.mode_leisure}
                agent.trips = {'work': agent.freq_work * 2, 'services': agent.freq_services * 2, 'leisure': agent.freq_leisure * 2}
                agent.length = {'work': agent.duration_work, 'services': agent.duration_services, 'leisure': agent.duration_leisure}
                agent.mode_trips = {m: 0 for m in self.tr_modes}
                for mode in self.tr_modes:
                    for dest in self.dests:
                        if agent.modes[dest] == mode:
                            agent.mode_trips[mode] += agent.trips[dest]
                agent.total_trips = sum(agent.trips.values())
                if agent.total_trips > 0:
                    agent.mode_shares = {k: v / agent.total_trips for k, v in agent.mode_trips.items()}
                else:
                    agent.mode_shares = {k: 0 for k, v in agent.mode_trips.items()}
                # Store original number of bike/walk trips
                agent.bike0 = agent.mode_trips['bike']
                agent.walk0 = agent.mode_trips['walk']
                # Create holders for utilities, probabilities
                agent.U = {mode: {'work': 0, 'services': 0, 'leisure': 0}
                                  for mode in self.tr_modes}
                agent.raw_prob = {mode: {'work': 0, 'services': 0, 'leisure': 0}
                                  for mode in self.tr_modes}
                agent.prob = {mode: {'work': 0, 'services': 0, 'leisure': 0}
                                  for mode in self.tr_modes}
            # Assign peers
            for agent in self.agents:
                # Find potential peers
                potential_peers = cluster_dict[agent.cluster]
                # Get peers randomly
                peers = np.random.choice(np.array(potential_peers), self.num_peers)
                agent.peers = peers

                
            # print(f"Agent {agent.h_id}: Income={agent.income}, Gas={agent.gas}, Awareness={agent.aware}, Group={agent.h_group}")

    def assign_attributes(self, agent, population, setup_vars):
        # Allocate a simulated response to agent
        rn = np.random.choice(population.index)
        agent_attr = population.loc[rn]
        for attr in setup_vars.index:
            if attr in agent_attr.index:
                var = agent_attr[attr]
                setattr(agent, attr, var)

    def step(self):
        self.year += 1
        print('-------------------------------------')
        print(self.year)
        # print(f"Simulation step for year {self.year} started.")
        # self.schedule.step()
        self.agents.shuffle_do("step")
        # self.recall_memory()
        # self.debug()
        # self.update_info()
        if self.map_update == "Yes":
            self.update_map()

        self.consideration()
        self.motivation()
        self.knowledge()
        self.utility()
        self.action()
        # self.save_energy()
        # self.invest()
        if self.year >= self.simulation_start: 
            self.learn()
            self.update_income()
        # self.update_energy()
        self.update_memory()
        self.print_summary()
        if self.year >= self.start_year + 3:
            self.datacollector.collect(self)

    def recall_memory(self):
        if self.year == start_year:
            for agent in self.agents:
                if random.uniform(0, 100) <= 2.5:
                    agent.act1 = True
                    agent.action += 1
                    agent.invest1 = True
                else:
                    agent.act1 = False
                    agent.invest1 = False                    

            # print(f"Memory recalled for year {self.year}.")

    def debug(self):
        if self.debugfiles:
            with open("output/debug.csv", "a") as file:
                for agent in self.agents:
                    file.write(
                        f"{self.year},{agent.h_id},{agent.income},{agent.gender},{agent.education},{agent.age},{agent.kn_aw},{agent.guilt},{agent.pn},{agent.sn},{agent.mot},{agent.pbc},{agent.pbcC},{agent.pbcS},{agent.cons},{agent.U}\n")
            # print("Debug information written to file.")

    def knowledge(self):
        for agent in self.agents:
            agent.guilt = "L" if agent.kn_aw < 4 else "H"
            # if agent.guilt == "H":
            #     agent.know = agent.aware / 7
        # print("Knowledge updated.")

    def motivation(self):
        for agent in self.agents:
            if agent.guilt == "H":
                agent.pn_ch += 0.7191 * agent.kn_aw_ch + 0.1558 * agent.sn_ch
                # Restrict change so pn cannot be higher than 5
                agent.pn_ch = agent.pn_ch - np.maximum(0, agent.pn + agent.pn_ch - 5)
                agent.pn += agent.pn_ch
                
                agent.mot = "L" if (agent.pn < 3.5) else "H"
            else:
                agent.mot = "L"
            agent.kn_aw_ch = 0
            agent.sn_ch = 0
        # print("Motivation updated.")

    def consideration(self):
        for agent in self.agents:
            if agent.mot == "H":
                agent.pbc_ch += 0.4489 * agent.pn_ch
                # Restrict change so pbc cannot be higher than 5
                agent.pbc_ch = agent.pbc_ch - np.maximum(0, agent.pbc + agent.pbc_ch - 5)
                agent.pbc += agent.pbc_ch
                agent.cons = "L" if (agent.pbc < 3.5) else "H"
            else:
                agent.cons = "L"
            agent.pn_ch = 0
            agent.pbc_ch = 0
        # print("Consideration updated.")

    def utility(self):
        # Get the infrastructure developments for the given year
        # Get previous year
        distr_infr_y_2024 = self.district_infra[['BEZNR', 'Infrastructure', '2024']].set_index(['BEZNR', 'Infrastructure']).copy()
        # get this year
        distr_infr_y = self.district_infra[['BEZNR', 'Infrastructure', str(self.year)]].set_index(['BEZNR', 'Infrastructure']).copy()
        # Difference (therefore development)
        distr_infr_diff = distr_infr_y[str(self.year)] - distr_infr_y_2024['2024']
        print('Infrastructure development vs. 2024')
        print(distr_infr_diff.groupby('Infrastructure').mean())
        for agent in self.agents:
            if agent.cons == "H":
                # Calculate utility and probability for all destinations
                for dest in self.dests: 
                    for mode in self.tr_modes: 
                        # Create key
                        key = mode + "_" + dest
                        # Get coefficients 
                        c = self.coefs[key]
                        # Gather data
                        s = pd.Series(0.0, index = c.index)
                        for var in s.index:
                            if var == 'const':
                                s[var] = 1.0
                            elif var in ['bikelane', 'resid_street', 'park']:
                                distr = agent.district
                                s[var] = distr_infr_diff.loc[(distr, var)]
                            else:
                                s[var] = getattr(agent, var)
                        # Calculate logit
                        u = (c * s).sum()
                        # print(c)
                        # print(s)
                        # Set attribute
                        agent.U[mode][dest] = u
                        # Calculate probability
                        raw_p = np.exp(u) / (1 + np.exp(u))
                        # Set attribute
                        agent.raw_prob[mode][dest] = raw_p
                    # Once all raw probabilities are calculated, normalise them to 1
                    prob_sum = (agent.raw_prob['bike'][dest] + agent.raw_prob['walk'][dest] + 
                                agent.raw_prob['car'][dest] + agent.raw_prob['pt'][dest])
                    agent.prob['bike'][dest] = agent.raw_prob['bike'][dest] / prob_sum
                    agent.prob['walk'][dest] = agent.raw_prob['walk'][dest] / prob_sum
                    agent.prob['car'][dest] = agent.raw_prob['car'][dest] / prob_sum
                    agent.prob['pt'][dest] = agent.raw_prob['pt'][dest] / prob_sum

        # print("Utility calculated.")



    def action(self):
        # Consider car, public transport, or motor trips
        # which can be replaced by 20-minute walk/biking
        for agent in self.agents:
            if agent.cons == "H":
                if agent.shift_mode == 0:
                    for d in self.dests:
                        if agent.prob['bike'][d] > 0.15 and not agent.prob['walk'][d] > 0.3:
                            # Create mode dictionary without bikes
                            modes_no_bike = agent.mode_trips.copy()
                            modes_no_bike.pop('bike', None)
                            # Replaceable trips
                            replace_bike = 0
                            # Find trips where not bike is used
                            if agent.modes[d] != 'bike':
                                # Assess length by mode
                                if agent.modes[d] == 'car':
                                    if agent.length[d] < 30:
                                        replace_bike += agent.trips[d]
                                if agent.modes[d] == 'pt':
                                    if agent.length[d] < 40:
                                        replace_bike += agent.trips[d]
                                # if agent.modes[d] == 'walk':
                                #     if agent.length[d] < 40:
                                #         replace_bike += agent.trips[d]
        
                            # Probability of replacing 1 trip
                            # Generate random numbers to see how many trips are replaced
                            random_nr = np.random.uniform(size = int(replace_bike), low = 0, high = 1)  
                            replaced_nr = sum(random_nr < agent.prob['bike'][d])
                            if replaced_nr > replace_bike / 2:
                                # If agent would choice bike in most cases it replaces all its trips to a destination 
                                agent.modes[d] = 'bike'
                            # Calculate which trips are replaced
                            # self.replace_trips(agent, modes_no_bike, replaced_nr)
                            agent.shift_mode = 1
                        elif agent.prob['walk'][d] > 0.3 and not agent.prob['bike'][d] > 0.1:
                            # Create mode dictionary without walking
                            modes_no_bike = agent.mode_trips.copy()
                            modes_no_bike.pop('walk', None)
                            # Replaceable trips
                            replace_walk = 0
                            # Find trips where not bike is used
                            if agent.modes[d] != 'walk':
                                # Assess length by mode
                                if agent.modes[d] == 'car':
                                    if agent.length[d] < 15:
                                        replace_walk += agent.trips[d]
                                if agent.modes[d] == 'pt':
                                    if agent.length[d] < 20:
                                        replace_walk += agent.trips[d]
                                # if agent.modes[d] == 'bike':
                                #     if agent.length[d] < 20:
                                #         replace_walk += agent.trips[d]
        
                            # Probability of replacing 1 trip
                            # Generate random numbers to see how many trips are replaced
                            random_nr = np.random.uniform(size = int(replace_walk), low = 0, high = 1)  
                            replaced_nr = sum(random_nr < agent.prob['walk'][d])
                            if replaced_nr > replace_walk / 2:
                                # If agent would choice bike in most cases it replaces all its trips to a destination 
                                agent.modes[d] = 'walk'
                            # Calculate which trips are replaced
                            # self.replace_trips(agent, modes_no_bike, replaced_nr)
                            agent.shift_mode = 1
                        elif agent.prob['walk'][d] > 0.3 and agent.prob['bike'][d] > 0.1:
                            # Create mode dictionary without walking
                            modes_no_active = agent.mode_trips.copy()
                            modes_no_active.pop('bike', None)
                            modes_no_active.pop('walk', None)
                            # Replaceable trips
                            # Assess walking
                            replace_walk = 0
                            # Find trips where not bike is used
                            if (agent.modes[d] != 'walk') & (agent.modes[d] != 'bike'):
                                # Assess length by mode
                                if agent.modes[d] == 'car':
                                    if agent.length[d] < 15:
                                        replace_walk += agent.trips[d]
                                if agent.modes[d] == 'pt':
                                    if agent.length[d] < 20:
                                        replace_walk += agent.trips[d]                    
                            # Probability of replacing 1 trip
                            # Generate random numbers to see how many trips are replaced
                            random_nr_walk = np.random.uniform(size = int(replace_walk), low = 0, high = 1)  
                            replaced_nr_walk = sum(random_nr_walk < agent.prob['walk'][d])
        
                            # Assess cycling
                            replace_bike = 0
                            # Find trips where not bike is used
                            if agent.modes[d] != 'bike':
                                # Assess length by mode
                                if agent.modes[d] == 'car':
                                    if agent.length[d] < 30:
                                        replace_bike += agent.trips[d]
                                if agent.modes[d] == 'pt':
                                    if agent.length[d] < 40:
                                        replace_bike += agent.trips[d]
                            # Probability of replacing 1 trip
                            # Generate random numbers to see how many trips are replaced
                            random_nr_bike = np.random.uniform(size = int(replace_bike), low = 0, high = 1)  
                            replaced_nr_bike = sum(random_nr_bike < agent.prob['bike'][d])
        
                            random_walk_or_bike = np.random.uniform(size = 1, low = 0, high = 1)
                            # Decide randomly between walking and cycling if both are preferred in most trips
                            # Weight probabilities based on agent probs
                            walk_vs_bike = agent.prob['walk'][d] / (agent.prob['walk'][d] + agent.prob['bike'][d])
                            if (replaced_nr_walk > replace_walk / 2) & (replaced_nr_bike > replace_bike / 2):
                                if (random_walk_or_bike < walk_vs_bike):
                                    agent.modes[d] = 'walk'
                                else:
                                    agent.modes[d] = 'bike'
                            # If only walking is preferred set to walking
                            elif (replaced_nr_walk > replace_walk / 2):
                                # If agent would choose bike in most cases it replaces all its trips to a destination 
                                agent.modes[d] = 'walk'
                            # If only cycling is preferred set to cycling
                            elif (replaced_nr_bike > replace_bike / 2):
                                # If agent would choose bike in most cases it replaces all its trips to a destination 
                                agent.modes[d] = 'bike'
                            else:
                                continue
                                
                            agent.shift_mode = 1
        
                    # Recalculate mode shares
                    agent.mode_trips = {m: 0 for m in self.tr_modes}
                    for mode in self.tr_modes:
                        for dest in self.dests:
                            if agent.modes[dest] == mode:
                                agent.mode_trips[mode] += agent.trips[dest]
                    if agent.total_trips > 0:
                        agent.mode_shares = {k: v / agent.total_trips for k, v in agent.mode_trips.items()}
                    else:
                        agent.mode_shares = {k: 0 for k, v in agent.mode_trips.items()}            
        # print("Actions determined for agents.")

    # def replace_trips(self, agent, replace_dict, replaced_nr):
    #     if replaced_nr > 0:
    #         # Replace Nans if any
    #         replace_dict = {k: v if not np.isnan(v) else 0 for k,v in replace_dict.items()}
    #         # Assume that trips are replaced with active modes proportionally by mode using probability distribution
    #         total_replace = sum(replace_dict.values())
    #         random_pool = []
    #         for k, v in replace_dict.items():
    #             random_pool = random_pool + [k] * int(round(v, 0))
    #         if total_replace > 0:
    #             # Adjust for inconsistencies in survey responses
    #             if replaced_nr > len(random_pool):
    #                 replaced_nr = len(random_pool)
    #             replaced_trips = np.array(random.sample(random_pool, replaced_nr))
    #             # Remove replaced trips from mode dictionary
    #             agent.modes = {k: v - (replaced_trips == k).sum() for k, v in agent.modes.items()}
        
    # def save_energy(self):
    #     gas_total = sum([agent.gas for agent in self.agents])

    #     if self.year >= 2017:
    #         for agent in self.agents:
    #             if agent.act1:
    #                 agent.save_a1 = agent.gas * 0.2
    #                 agent.gas -= agent.save_a1
    #             else:
    #                 agent.save_a1 = 0

    #     save_gas = sum([agent.save_a1 for agent in self.agents])
    #     # self.save_gas_com += save_gas
    #     print(f"Energy savings calculated for year {self.year}.")


    def learn(self):
        if self.year >= 2011:
            for agent in self.agents:
                if self.learning == "No":
                    continue
                # if self.learning in ["Slow dynamics", "Fast dynamics"]:
                #     if agent.act1 or agent.invest1:
                #         if agent.pbc < 4.6:
                #             agent.pbc += agent.pbc * 0.05

                #         neighbors = self.grid.get_neighbors(agent.pos, moore=True, include_center=False)
                #         if len(neighbors) > 0:
                #             ngb_k_mean = sum([n.kn_aw for n in neighbors]) / len(neighbors)
                #             ngb_k_median = sorted([n.kn_aw for n in neighbors])[len(neighbors) // 2]
                #             ngb_k = max(ngb_k_mean, ngb_k_median)
    
                #             ngb_pn_mean = sum([n.pn for n in neighbors]) / len(neighbors)
                #             ngb_pn_median = sorted([n.pn for n in neighbors])[len(neighbors) // 2]
                #             ngb_pn = max(ngb_pn_mean, ngb_pn_median)
    
                #             ngb_sn_mean = sum([n.sn for n in neighbors]) / len(neighbors)
                #             ngb_sn_median = sorted([n.sn for n in neighbors])[len(neighbors) // 2]
                #             ngb_sn = max(ngb_sn_mean, ngb_sn_median)
    
                #             ngb_pbc_mean = sum([n.pbc for n in neighbors]) / len(neighbors)
                #             ngb_pbc_median = sorted([n.pbc for n in neighbors])[len(neighbors) // 2]
                #             ngb_pbc = max(ngb_pbc_mean, ngb_pbc_median)

                #         if len(neighbors) > 4:
                #             for n in neighbors:
                #                 if n.kn_aw < ngb_k and n.kn_aw < 4.6:
                #                     n.kn_aw_ch = n.kn_aw * 0.05
                #                     n.kn_aw += n.kn_aw_ch
                #                 if n.pn < ngb_pn and n.pn < 4.6:
                #                     n.pn_ch = n.pn * 0.05
                #                     n.pn += n.pn_ch
                #                 if n.sn < ngb_sn and n.sn < 4.6:
                #                     n.sn_ch = n.sn * 0.05
                #                     n.sn += n.sn_ch
                #                 if n.pbc < 4.5 and n.pbc < 4.6:
                #                     n.pbc_ch = n.pbc * 0.05
                #                     n.pbc += n.pbc_ch

                if self.learning == "Informative":
                    
    
                    # if agent.mode_trips['bike'] < 4 or agent.mode_trips['walk'] < 4:
                    # Learning from neighbors  
                    neighbors = self.grid.get_neighbors(agent.pos, moore=True, include_center=False, radius = 25)
                    active_neighbors = [n for n in neighbors if n.mode_trips['bike'] + n.mode_trips['walk'] > 7]
                    if len(active_neighbors) > 0:
                    
                        ngb_k_mean = sum([n.kn_aw for n in active_neighbors]) / len(active_neighbors)
                        ngb_k_median = sorted([n.kn_aw for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_k = max(ngb_k_mean, ngb_k_median)

                        ngb_pn_mean = sum([n.pn for n in active_neighbors]) / len(active_neighbors)
                        ngb_pn_median = sorted([n.pn for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_pn = max(ngb_pn_mean, ngb_pn_median)

                        ngb_sn_mean = sum([n.sn for n in active_neighbors]) / len(active_neighbors)
                        ngb_sn_median = sorted([n.sn for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_sn = max(ngb_sn_mean, ngb_sn_median)

                        ngb_pbc_mean = sum([n.pbc for n in active_neighbors]) / len(active_neighbors)
                        ngb_pbc_median = sorted([n.pbc for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_pbc = max(ngb_pbc_mean, ngb_pbc_median)

                        if agent.kn_aw < ngb_k and agent.kn_aw < 4.9:
                            agent.kn_aw_ch = agent.kn_aw * 0.05
                            agent.kn_aw += agent.kn_aw_ch
                        if agent.guilt == 'H' and agent.pn < ngb_pn and agent.pn < 4.9:
                            agent.pn_ch = agent.pn * 0.05
                            agent.pn += agent.pn_ch 
                        if agent.guilt == 'H' and agent.sn < ngb_sn and agent.sn < 4.9:
                            agent.sn_ch = agent.sn * 0.05
                            agent.sn += agent.sn_ch
                        if agent.mot == 'H' and agent.pbc < ngb_pbc and agent.pbc < 4.9:
                            agent.pbc_ch = agent.pbc * 0.05
                            agent.pbc += agent.pbc_ch

                if self.learning == "Peers":
                    
                    # if agent.mode_trips['bike'] < 4 or agent.mode_trips['walk'] < 4:
                    # Learning from neighbors  
                    peers = agent.peers
                    active_neighbors = [self.agents[n] for n in peers if self.agents[n].mode_trips['bike'] + self.agents[n].mode_trips['walk'] > 7]
                    if len(active_neighbors) > 0:
                    
                        ngb_k_mean = sum([n.kn_aw for n in active_neighbors]) / len(active_neighbors)
                        ngb_k_median = sorted([n.kn_aw for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_k = max(ngb_k_mean, ngb_k_median)

                        ngb_pn_mean = sum([n.pn for n in active_neighbors]) / len(active_neighbors)
                        ngb_pn_median = sorted([n.pn for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_pn = max(ngb_pn_mean, ngb_pn_median)

                        ngb_sn_mean = sum([n.sn for n in active_neighbors]) / len(active_neighbors)
                        ngb_sn_median = sorted([n.sn for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_sn = max(ngb_sn_mean, ngb_sn_median)

                        ngb_pbc_mean = sum([n.pbc for n in active_neighbors]) / len(active_neighbors)
                        ngb_pbc_median = sorted([n.pbc for n in active_neighbors])[len(active_neighbors) // 2]
                        ngb_pbc = max(ngb_pbc_mean, ngb_pbc_median)

                        if agent.kn_aw < ngb_k and agent.kn_aw < 4.9:
                            agent.kn_aw_ch = agent.kn_aw * 0.05
                            agent.kn_aw += agent.kn_aw_ch
                        if agent.guilt == 'H' and agent.pn < ngb_pn and agent.pn < 4.9:
                            agent.pn_ch = agent.pn * 0.05
                            agent.pn += agent.pn_ch 
                        if agent.guilt == 'H' and agent.sn < ngb_sn and agent.sn < 4.9:
                            agent.sn_ch = agent.sn * 0.05
                            agent.sn += agent.sn_ch
                        if agent.mot == 'H' and agent.pbc < ngb_pbc and agent.pbc < 4.9:
                            agent.pbc_ch = agent.pbc * 0.05
                            agent.pbc += agent.pbc_ch

                # if self.learning == "Informative-Soft":
                #     if agent.kn_aw <= 4.6:
                #         agent.kn_aw += agent.kn_aw * 0.05

                # if self.learning == "Promoting":
                #     if agent.kn_aw <= 4.6:
                #         agent.kn_aw += agent.kn_aw * 0.05
        # print("Learning process updated.")

    def update_income(self):
        for agent in self.agents:
            rn = random.uniform(0, 100)
            if rn >= 50:
                agent.income = agent.income * 1.02
            else:
                continue
        # print("Income updated.")

    def update_energy(self):
        if self.year >= 2017:
            for agent in self.agents:
                if agent.act1 and agent.dw_elab >= 2:
                    agent.dw_elab -= 1
        # print("Energy status updated.")

    def update_memory(self):
        for agent in self.agents:
            if agent.shift_mode == 1:
                agent.shift_year += 1
            if agent.shift_year >= 3:
                agent.shift_mode = 0
                agent.shift_year = 0

        # print("Memory updated.")

    def update_map(self):
        # Load new map in given years
        if self.year in [2025, 2030, 2035]:
            print('Update map')
            if self.year == 2025:
                # Housing map in 2025
                vienna_housing_map = gpd.read_file('input//vienna_houses//vienna_social_houses_distances_2025.shp')
            if self.year == 2030:
                # Housing map in 2030
                vienna_housing_map = gpd.read_file('input//vienna_houses//vienna_social_houses_distances_2030.shp')
            if self.year == 2035:
                # Housing map in 2035
                vienna_housing_map = gpd.read_file('input//vienna_houses//vienna_social_houses_distances_2035.shp')
            distances = ['bike_dist', 'green_dist', 'hike_dist', 'resid_dist', 'slow_dist']
            facilities = ['bike_bool', 'green_bool', 'hike_bool', 'resid_bool', 'slow_bool']
            housing_distances = vienna_housing_map[distances]
            housing_facilities = vienna_housing_map[facilities]
            # Update agents
            for agent in self.agents:
                # New facilities
                facilities = housing_facilities.loc[agent.house_id].to_dict(orient = 'list')
                # Increase motivation if facility is not available near the agent
                # print(facilities['green_bool'][0],agent.facilities['green_bool'][0]) 
                # print((facilities['green_bool'][0] > 0))
                # print((agent.facilities['green_bool'][0] < 1))
                if (facilities['bike_bool'][0] > 0) and (agent.facilities['bike_bool'][0] < 1):
                    if agent.pn < 4.5:
                        agent.pn_ch += 0.7
                        agent.pn += agent.pn_ch
                    if agent.pbc < 4.5:
                        agent.pbc_ch += 0.7
                        agent.pbc += agent.pbc_ch
                if (facilities['hike_bool'][0] > 0) and (agent.facilities['hike_bool'][0] < 1):
                    if agent.kn_aw < 4.5:
                        agent.kn_aw_ch += 0.7
                        agent.kn_aw += agent.kn_aw_ch
                if (facilities['resid_bool'][0] > 0) and (agent.facilities['resid_bool'][0] < 1):
                    if agent.pn < 4:
                        agent.pn_ch += 0.9
                        agent.pn += agent.pn_ch
                    if agent.pbc < 4.5:
                        agent.pbc_ch += 0.7
                        agent.pbc += agent.pbc_ch
                if (facilities['green_bool'][0] > 0) and (agent.facilities['green_bool'][0] < 1):
                    if agent.kn_aw < 4.5:
                        agent.kn_aw_ch += 0.5
                        agent.kn_aw += agent.kn_aw_ch
                    if agent.pbc < 4.5:
                        agent.pbc_ch += 0.7
                        agent.pbc += agent.pbc_ch
                        
                # Update houseing characteristics to agent (distances from facilities)
                agent.distances = housing_distances.loc[agent.house_id].to_dict(orient = 'list')
                agent.facilities = facilities
        # # Load new map in 2035
        # if self.year == 2035:
        #      # Housing map in 2035
        #     vienna_housing_map = gpd.read_file('input//vienna_houses//vienna_social_houses_distances_2035_3_4_5_10_bezirk.shp')
        #     distances = ['bike_dist', 'green_dist', 'hike_dist', 'resid_dist', 'slow_dist']
        #     facilities = ['bike_bool', 'green_bool', 'hike_bool', 'resid_bool', 'slow_bool']
        #     housing_distances = vienna_housing_map[distances]
        #     housing_facilities = vienna_housing_map[facilities]
        #     # Update agents
        #     for agent in self.agents:
        #         # New facilities
        #         facilities = housing_facilities.loc[agent.house_id].to_dict(orient = 'list')
        #         # Increase perceived behaviour if facility is not available near the agent
        #         if agent.pbc < 4.6 and agent.facilities['bike_bool'][0] == 0 and facilities['bike_bool'][0] == 1:
        #             agent.pbc += agent.pbc * 0.1
        #         if agent.pbc < 4.6 and agent.facilities['green_bool'][0] == 0 and facilities['green_bool'][0] == 1:
        #             agent.pbc += agent.pbc * 0.1
        #         if agent.pbc < 4.6 and agent.facilities['slow_bool'][0] == 0 and facilities['slow_bool'][0] == 1:
        #             agent.pbc += agent.pbc * 0.1
        #         if agent.pbc < 4.6 and agent.facilities['hike_bool'][0] == 0 and facilities['hike_bool'][0] == 1:
        #             agent.pbc += agent.pbc * 0.1
        #         if agent.pbc < 4.6 and agent.facilities['resid_bool'][0] == 0 and facilities['resid_bool'][0] == 1:
        #             agent.pbc += agent.pbc * 0.1
        #         # Update houseing characteristics to agent (distances from facilities)
        #         agent.distances = housing_distances.loc[agent.house_id].to_dict(orient = 'list')
        #         agent.facilities = facilities


        # print("Housing information updated.")

    def update_info(self):
        for agent in self.agents:
            agent.act1 = False
        # print("Agent info updated.")

    def print_summary(self):
        # Print summary statistics for the current year
        num_agents_bike = sum(agent.mode_trips['bike'] for agent in self.agents)
        num_agents_walk = sum(agent.mode_trips['walk'] for agent in self.agents)
        num_agents_car = sum(agent.mode_trips['car'] for agent in self.agents)
        num_agents_pt = sum(agent.mode_trips['pt'] for agent in self.agents)
        total = sum(agent.total_trips for agent in self.agents)

        guilt = sum(1 for agent in self.agents if agent.guilt == 'H')
        mot = sum(1 for agent in self.agents if agent.mot == 'H')
        cons = sum(1 for agent in self.agents if agent.cons == 'H')

        kn_aw = np.average([agent.kn_aw for agent in self.agents])
        pn = np.average([agent.pn for agent in self.agents])
        pbc = np.average([agent.pbc for agent in self.agents])
        u_bike = np.average([agent.U['bike']['work'] for agent in self.agents])
        u_walk = np.average([agent.U['walk']['work'] for agent in self.agents])

        # if self.year == 2040:
        
        dists = set()
        dists = [agent.district for agent in self.agents if agent.district not in dists and (dists.add(agent.district) or True)]
        district = {"Bicycle": {d: sum(agent.mode_trips['bike'] for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "Walk": {d: sum(agent.mode_trips['walk'] for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "Car": {d: sum(agent.mode_trips['car'] for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "PT": {d: sum(agent.mode_trips['pt'] for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "PBC": {d: sum(agent.pbc for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "PN": {d: sum(agent.pn for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "KN": {d: sum(agent.kn_aw for agent in self.agents if agent.district == d) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "guilt": {d: sum(1 for agent in self.agents if (agent.guilt == 'H') & (agent.district == d)) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "mot": {d: sum(1 for agent in self.agents if (agent.mot == 'H') & (agent.district == d)) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "cons": {d: sum(1 for agent in self.agents if (agent.cons == 'H') & (agent.district == d)) / 
                                sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "district_pop": {d: sum(1 for agent in self.agents if agent.district == d)
                                                for d in dists},
                    "total_trips": {d: sum(agent.total_trips for agent in self.agents if agent.district == d)
                                                for d in dists}}
        # print(district)
        # bike_district = pd.DataFrame.from_dict(district["Bicycle"], 
        #                                        orient='index', columns=['bike_trips']
        #                                       ).reset_index().rename(columns={'index': 'district'}).sort_values('district')
        # bike_fn = "output/results/fig_" + str(self.iter) + "_" + self.scenario + "_bike_" + str(self.year) + ".csv"
        # bike_district.to_csv(bike_fn, index = False)

        # walk_district = pd.DataFrame.from_dict(district["Walk"], 
        #                                        orient='index', columns=['bike_trips']
        #                                       ).reset_index().rename(columns={'index': 'district'}).sort_values('district')
        # walk_fn = "output/results/fig_" + str(self.iter) + "_" + self.scenario + "_walk_" + str(self.year) + ".csv"
        # walk_district.to_csv(walk_fn, index = False)

        out_df = pd.DataFrame(district)
        out_df = out_df.reset_index().rename(columns={'index': 'district'}).sort_values('district')
        ou_fn = "output/results/" + self.scenario + "_iter" + str(self.iter) + "_" + str(self.year) + ".csv"
        out_df.to_csv(ou_fn, index = False)
        
        print("------------")
        print(f"Year: {self.year}")
        # print(f"Number of cycling trips: {round(num_agents_bike, 0)}")
        print(f"Share of cycling trips: {round(num_agents_bike / total, 3)}")
        # print(f"Number of walking trips: {round(num_agents_walk, 0)}")
        print(f"Share of walking trips: {round(num_agents_walk / total, 3)}")
        print(f"Share of car trips: {round(num_agents_car / total, 3)}")
        print(f"Share of public tr. trips: {round(num_agents_pt / total, 3)}")

        print(f"Knowledge & Awareness: {round(kn_aw, 3)}")
        print(f"Personal norm: {round(pn, 3)}")
        print(f"PBC: {round(pbc, 3)}")
        print(f"Guilt: {guilt}")
        print(f"Motivation: {mot}")
        print(f"Consideration: {cons}")
        print(f"Utility bicycle: {round(u_bike, 3)}")
        print(f"Utility walking: {round(u_walk, 3)}")


In [4]:
# Set portrayal

def agent_portrayal(agent):
    if agent.mode_trips['bike'] + agent.mode_trips['walk'] > 10:
        size = 60
        color = "tab:red"
    else: 
        size = 50
        color = "tab:blue"
    return {
        "color": color,
        "size": size,
    }


In [5]:
# Main function

def main(iters):
    # random.seed(123)
    population = pd.read_csv('input//simulated_population.csv')
    vienna_housing_map = gpd.read_file('input//vienna_houses//vienna_social_houses_distances.shp')
    config = configparser.ConfigParser()
    config.read(r'settings.ini')
    # args = parse_args()
    start_year = config.getint('default', 'start_year')
    end_year = config.getint('default', 'end_year')
    seed = config.get('default', 'seed')
    timeline = list(range(end_year - start_year))
    district_infra_fn = config.get('default', 'infra_scenario') 
    district_infra = pd.read_csv('input//vienna_district_infra__{}.csv'.format(district_infra_fn))
    policy = config.get('default', 'policy')
    # Import coefficients
    coef_fn = os.listdir('output/coefficients')
    coefs = {}
    for fn in coef_fn:
        var = fn.split('__')
        if 'coefficients' in var[1]:
            c = pd.read_csv('output/coefficients/' + fn, index_col = 0)
            c = c['Coef.']
            if 'car' in var[0]:
                c['bikelane'] = -2
                c['resid_street'] = -100
                c['park'] = 0
            if 'bike' in var[0]:
                c['bikelane'] = 5
                c['resid_street'] = 100
                c['park'] = 50
            if 'walk' in var[0]:
                c['bikelane'] = 2
                c['resid_street'] = 150
                c['park'] = 50
            if 'pt' in var[0]:
                c['bikelane'] = 1
                c['resid_street'] = -10
                c['park'] = 20
            coefs[var[0]] = c

    print("-----------------------------------------")
    print("Initiate BENCH-Mobility model run")
    
    for learning in ['No', 'Informative', 'Peers']:
        scenario = config.get('default', 'scenario')
        print('--------------------------------------')
        print('')
        print('{0} scenario with {1} learning'.format(scenario, learning))
        print('')
        print('--------------------------------------')

        if learning == 'No':
            scenario_name = scenario + '_wo_l'
        if learning == 'Informative':
            scenario_name = scenario + '_w_il'
        if learning == 'Peers':
            scenario_name = scenario + '_w_pl'
            
        for i in range(iters):
            print("Model iteration: {0} / {1}".format(i + 1, iters))
            if seed is not None:
                random.seed(int(seed) + i)
                np.random.seed(int(seed) + i)
                
            model = CommunityModel(
                population,
                vienna_housing_map,
                coefs = coefs,
                district_infra = district_infra,
                start_year = start_year,
                simulation_start = config.getint('default', 'simulation_start'),
                num_agents = config.getint('default', 'num_agents'),
                num_peers = config.getint('default', 'num_peers'),
                case_study = config.get('default', 'case_study'),
                scenario = scenario_name,
                data = config.get('default', 'data'),
                learning = learning,
                map_update = config.get('default', 'map_update'),
                policy = policy,
                iteration = i
            )
            
            for _ in timeline:  # Simulate
                model.step()


In [6]:
# Run
if __name__ == "__main__":
    path = Path(sys.path[0])
    os.chdir(path)
    main(1)

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'C:\\Users\\torrenperaire\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip'

In [ ]:
## Graphical User Interface

In [ ]:
path = Path(sys.path[0])
os.chdir(path)
population = pd.read_csv('input//simulated_population.csv')
vienna_housing_map = gpd.read_file('input//vienna_houses//vienna_social_houses_distances.shp')


In [ ]:
config = configparser.ConfigParser()
config.read(r'settings.ini')
# args = parse_args()
start_year = config.getint('default', 'start_year')
end_year = config.getint('default', 'end_year')
seed = config.get('default', 'seed')
timeline = list(range(end_year - start_year))
if seed is not None:
    random.seed(seed)
model = CommunityModel(
    population,
    vienna_housing_map,
    start_year = start_year,
    simulation_start = config.getint('default', 'simulation_start'),
    num_agents = config.getint('default', 'num_agents'),
    num_peers = config.getint('default', 'num_peers'),
    case_study = config.get('default', 'case_study'),
    scenario = config.get('default', 'scenario'),
    data = config.get('default', 'data'),
    learning = config.get('default', 'learning'),
    infrastructure = config.get('default', 'infrastructure'))

In [ ]:
SpaceGraph = make_space_component(agent_portrayal, linewidth = 0.5)
BikePlot = make_plot_component("Bicycle")
WalkPlot = make_plot_component("Walk")
AwarenessPlot = make_plot_component("Awareness")
pbcPlot = make_plot_component("PerceivedBehaviourControl")
pnormPlot = make_plot_component("PersonalNorm")

model_params = {
    "num_agents": {
        "type": "SliderInt",
        "value": config.getint('default', 'num_agents'),
        "label": "Number of agents:",
        "min": 1000,
        "max": 3000,
        "step": 50,
    },
    "population": population,
    "vienna_housing_map": vienna_housing_map,
    "start_year" : start_year,
    "simulation_start" : config.getint('default', 'simulation_start'),
    # "num_agents" : config.get('default', 'num_agents'),
    "num_peers" : config.getint('default', 'num_peers'),
    "case_study" : config.get('default', 'case_study'),
    "scenario" : config.get('default', 'scenario'),
    "data" : config.get('default', 'data'),
    "learning" : config.get('default', 'learning'),
    "infrastructure" : config.get('default', 'infrastructure')}    


In [ ]:
page = SolaraViz(
        model,
        components=[SpaceGraph, BikePlot, WalkPlot, AwarenessPlot, pbcPlot, pnormPlot],
        model_params=model_params,
        name="BENCH",
)
# This is required to render the visualization in the Jupyter notebook
page